# Fine-tune GLM-4-9B-0414 (QLoRA) for coding
Base model: `zai-org/GLM-4-9B-0414` (MIT license). Runs on free Colab T4 (16GB)
using 4-bit QLoRA via `transformers` + `peft` + `bitsandbytes` (GLM4 is a
custom architecture not yet covered by Unsloth's fast kernels, so we use the
standard, well-supported HF stack here instead).

See `CLAUDE.md` in this folder for the priorities this pipeline follows:
size reduction first (without breaking capability), then dataset quality
across Go / Python / JS+React / PostgreSQL / system design.

Steps: install -> load base model in 4-bit -> attach LoRA -> train -> merge
-> convert to GGUF (q4_k_m) via llama.cpp for Ollama deployment.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

base_model = "zai-org/GLM-4-9B-0414"
max_seq_length = 2048

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Upload dataset.jsonl (from prepare_dataset.py) via the Colab file browser first.
# Dataset must cover Go / Python / JS+React / PostgreSQL / system-design per CLAUDE.md.
from datasets import load_dataset

prompt_template = """Below is an instruction that describes a coding task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for instruction, inp, output in zip(examples["instruction"], examples["input"], examples["output"]):
        text = prompt_template.format(instruction, inp, output) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

dataset = load_dataset("json", data_files="dataset.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched=True)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    args=SFTConfig(
        max_seq_length=max_seq_length,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        logging_steps=1,
        optim="paged_adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        packing=False,
    ),
)

trainer_stats = trainer.train()

## Merge LoRA into base weights, then convert to GGUF (q4_k_m) for Ollama
Per `CLAUDE.md`: q4_k_m is the safe default (~0.5% perplexity loss). Never go
below 4-bit (3-bit/2-bit) without recording a benchmark showing no real
coding-quality regression.

In [ ]:
merged_model = model.merge_and_unload()
merged_model.save_pretrained("glm4-coder-merged", safe_serialization=True)
tokenizer.save_pretrained("glm4-coder-merged")

In [ ]:
# llama.cpp has supported the GLM4 architecture since April 2025 (used by the
# community GGUF repos for GLM-4-9B-0414). Convert the merged fp16 model, then quantize.
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!pip install -q -r llama.cpp/requirements.txt
!python llama.cpp/convert_hf_to_gguf.py glm4-coder-merged --outfile glm4-coder-f16.gguf --outtype f16
!cmake -S llama.cpp -B llama.cpp/build -DGGML_CUDA=OFF && cmake --build llama.cpp/build --target llama-quantize -j
!./llama.cpp/build/bin/llama-quantize glm4-coder-f16.gguf glm4-coder-q4_k_m.gguf Q4_K_M

In [ ]:
# Optional: push the GGUF straight to your Hugging Face account so RunPod/Ollama can pull it directly
# from huggingface_hub import login, upload_file
# login(token="hf_...")
# upload_file(path_or_fileobj="glm4-coder-q4_k_m.gguf", path_in_repo="glm4-coder-q4_k_m.gguf", repo_id="YOUR_HF_USERNAME/glm4-coder")